# cat_game_rl on Google Colab

**What it does:** `cat_game_rl` learns to schedule crafting in a 23-item game (19 craftable + 4 base, 8064 ticks / episode). It compares **RL (MaskablePPO via sb3-contrib) + NSGA-II GA (DEAP)** optimizing 23-item crafting batch schedules, against order-based baselines. Config: `config/{training,targets,ga,crafting_tree}.yaml`; code: `src/{core,cat_game_env,agent,ga,dashboard}`; scripts: `scripts/{train,evaluate,run_ga,run_baselines,launch_dashboard}.py`.

**How to use:** Colab menu `Runtime > Change runtime type > T4 GPU (recommended)` > `Runtime > Run all`. Smoke cells run first (minutes); full 2M-timestep training / 500-gen GA cells are **guarded optional** and skipped by default.

**Repo/branch:** `https://github.com/ankithaldar/all_projects.git`, branch `cat_game_rl/ai_implementation`, subdir `cat_game_rl/`.

**Outputs:** `output/models/final_model_*.zip` (or `.zip` under `output/models`), `output/batch_schedule_*.txt`, `output/ga_results/ga_batch_schedule_*.txt`, `output/logs/`, TensorBoard logs.


> **Supply-chain note (SEC-05):** Before Run all, verify branch/commit before Run all; fallback pip is unpinned (SEC-05 accepted Colab risk). Check the clone cell's branch/commit above and pin it if needed.

In [ ]:
# Clone fresh each run: delete any existing clone, then re-clone (idempotent / re-runnable)
# Safe: only removes the exact 'all_projects' dir in the Colab cwd, nothing else.
!rm -rf all_projects
!git clone --branch cat_game_rl/ai_implementation --depth 1 https://github.com/ankithaldar/all_projects.git
%cd all_projects/cat_game_rl
!pwd && !git branch --show-current && !git log -1 --oneline


In [ ]:
!python --version
!nvidia-smi || true
import sys
print(sys.version)
try:
    import torch
    print('torch', torch.__version__, 'cuda_available', torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
except Exception as e:
    print('torch check failed (install next cell):', e)


In [ ]:
# Install: pinned requirements.txt (torch==2.2.2, numpy==1.26.4) may fail on Colab Python 3.12+
# (no wheels for old pins), so fall back to unpinned compatible versions capped at numpy<3.
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
if r.returncode != 0:
    print('Pinned install failed; installing compatible fallback...')
    fallback = ['numpy<3', 'gymnasium', 'stable-baselines3', 'sb3-contrib', 'torch', 'pyyaml', 'deap', 'streamlit', 'plotly', 'pandas', 'tensorboard', 'pytest', 'pytest-cov']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + fallback, check=False)
import importlib
for m in ['torch', 'numpy', 'gymnasium', 'stable_baselines3', 'sb3_contrib', 'yaml', 'deap']:
    try:
        mod = importlib.import_module(m)
        print(m, getattr(mod, '__version__', 'ok'))
    except Exception as e:
        print(m, 'MISSING', e)


In [ ]:
# Smoke: pytest (coverage gate in pyproject addopts is bypassed for smoke speed)
!python -m pytest tests/ -q -p no:cacheprovider -o addopts=''
# Tiny env sanity (replicates scripts/train.py sys.path + env_config construction)
import os, sys, yaml
sys.path.insert(0, os.getcwd())  # scripts do sys.path.insert(0, parent-of-scripts)
from src.core.items import CraftingTree
from src.cat_game_env.crafting_env import CraftingEnv
tree = CraftingTree.from_yaml('config/crafting_tree.yaml')
print('items in tree:', len(tree.topo_order))
with open('config/training.yaml') as f:
    cfg = yaml.safe_load(f)
env_cfg = {'crafting_tree_path': 'config/crafting_tree.yaml', 'targets_path': 'config/targets.yaml', 'max_batch_size': cfg['environment']['max_batch_size'], 'max_ticks': cfg['environment']['max_ticks'], 'initial_coins': cfg['environment']['initial_coins'], 'initial_stash': cfg['environment'].get('initial_stash', {}), 'reward_weights': cfg.get('reward_weights')}
env = CraftingEnv(env_cfg)
obs, _ = env.reset()
print('reset ok, obs keys:', list(obs.keys()) if isinstance(obs, dict) else type(obs))
for i in range(5):
    obs, reward, term, trunc, _ = env.step(env.action_space.sample())
    print(f'step {i}: reward={reward:.3f} term={term} trunc={trunc}')
print('env sanity PASS')


In [ ]:
# Train SMOKE (verified CLI: scripts/train.py --timesteps/--frame-skip/--config)
!python scripts/train.py --timesteps 5000
!ls -lh output/models/ 2>/dev/null || echo 'no models dir yet'


In [ ]:
# OPTIONAL FULL TRAINING - skipped by default so Run-All stays short (~minutes, not 30min+)
RUN_FULL_TRAINING = False  # set True to run full 2000000 timesteps (config training.total_timesteps)
if RUN_FULL_TRAINING:
    import subprocess, sys
    subprocess.run([sys.executable, 'scripts/train.py', '--timesteps', '2000000'], check=False)
else:
    print('Skipped full 2M training (RUN_FULL_TRAINING=False). Set True to enable.')


In [ ]:
# Evaluate / export (verified CLI: scripts/evaluate.py --model REQUIRED --config --output)
!ls -R output/models 2>/dev/null || echo 'output/models missing (train smoke first)'
import glob
models = sorted(glob.glob('output/models/*.zip'))
print('models found:', models)


In [ ]:
import glob, subprocess, sys
models = sorted(glob.glob('output/models/*.zip'))
if models:
    latest = models[-1]
    print('Evaluating', latest)
    subprocess.run([sys.executable, 'scripts/evaluate.py', '--model', latest], check=False)
else:
    print('No model found — skipping evaluate. Run train smoke cell first.')


In [ ]:
!ls -lh output/*.txt output/models/ 2>/dev/null || true
!head -n 20 output/batch_schedule*.txt 2>/dev/null || echo 'no batch_schedule yet'


In [ ]:
# GA smoke (verified CLI: scripts/run_ga.py --generations/--config/--output-dir)
!python scripts/run_ga.py --generations 5
!ls -lh output/ga_results/ 2>/dev/null || true
!head -n 20 output/ga_results/ga_batch_schedule*.txt 2>/dev/null || echo 'no GA schedule yet'


In [ ]:
# OPTIONAL FULL GA - skipped by default (config ga.n_generations=500)
RUN_FULL_GA = False  # set True to run full 500 generations
if RUN_FULL_GA:
    import subprocess, sys
    subprocess.run([sys.executable, 'scripts/run_ga.py', '--generations', '500'], check=False)
else:
    print('Skipped full GA (RUN_FULL_GA=False). Set True to enable.')


In [ ]:
# Dashboard (optional, non-blocking): verified entry scripts/launch_dashboard.py
# It runs: python -m streamlit run src/dashboard/app.py (no CLI args).
# In Colab, Streamlit needs port forwarding / `streamlit run --server.port 8501`.
RUN_DASHBOARD = False  # set True to launch in background
if RUN_DASHBOARD:
    pass  # uncomment next line manually if you want it:
    # !nohup python scripts/launch_dashboard.py > dashboard.log 2>&1 &
    # !sleep 3 && !cat dashboard.log
else:
    print('Dashboard skipped. To launch: nohup python scripts/launch_dashboard.py > dashboard.log 2>&1 & then expose port 8501.')


## Outputs / Artifacts
- Models: `output/models/final_model_<timestamp>` (`.zip`) + checkpoints under `output/models/`
- Schedules: `output/batch_schedule_<timestamp>.txt` (RL evaluate), `output/ga_results/ga_batch_schedule_<timestamp>.txt` (GA)
- Logs: `output/logs/` (TensorBoard: `tensorboard --logdir output/logs`), `output/ga_results/ga_population_log.jsonl`

Download artifacts: in Colab Files panel, or `!zip -r outputs.zip output && !ls -lh outputs.zip`.

## Troubleshooting
- **Pinned deps fail on Python 3.12+:** install cell already falls back to unpinned `torch` + `numpy<3`; restart runtime (`Runtime > Restart session`) and re-run install cell.
- **No GPU / `nvidia-smi` fails:** training still runs on CPU but slower; select T4 via `Runtime > Change runtime type`.
- **Wrong branch / stale code:** `!git branch --show-current` should print `cat_game_rl/ai_implementation`; if not, delete `all_projects/` and re-run clone cell.
- **Tests fail on coverage gate:** smoke uses `-o addopts=''` to bypass `--cov-fail-under=90`; run without it locally for the full gate.
- **Evaluate says no model:** run train smoke first; evaluate requires a `.zip` under `output/models/`.

## Cleanup
`!rm -rf all_projects` (only inside Colab VM — your repo copy is unaffected).
